# rotorwave — wave analysis of rotors with longitudinal periodicity

Companion notebook to
*Lamas & Nicoletti, Wave analysis of rotors with longitudinal periodicity*,
J. Sound Vib. **571** (2024) 118095.

It walks through the two wave problems the paper solves, checks them against
the frequency response of the finite rotor, and ends with two analyses that go
beyond the original figures.

The first cell works **with or without the package installed**: if it is not
installed, it adds the `src` folder to the Python path.


In [ ]:
import sys
from pathlib import Path
import importlib.util

if importlib.util.find_spec('rotorwave') is None:
    src = Path.cwd() / 'src'
    if not src.is_dir():
        src = Path.cwd().parent / 'src'   # in case the notebook sits in a subfolder
    sys.path.insert(0, str(src))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import rotorwave
print('rotorwave', rotorwave.__version__)

## 1. The rotor

A 1500 mm steel shaft, 100 mm in diameter, carrying 11 equally spaced disks of
380 x 22 mm — the case study of Section 3 of the paper.

Because the disks are equally spaced, the rotor is a periodic structure: it is
built by repeating one **unit cell** (a piece of shaft plus one disk) eleven
times.  The whole wave analysis is done on that single cell.

In [ ]:
from rotorwave import reference_rotor

rotor = reference_rotor(n_disks=11, elements_per_cell=40)
print(rotor.summary())

cell = rotor.unit_cell()
print('unit cell:', cell.n_dofs, 'degrees of freedom,',
      f'{cell.cell_length*1e3:.1f} mm long')

## 2. The $\omega(k)$ problem — dispersion diagram and band gaps

Prescribe a real wavenumber $k$ and compute the frequencies that propagate with
it.  Sweeping $k$ over the irreducible Brillouin zone, $0 \le k\Delta/\pi \le 1$,
traces the dispersion branches.

Each branch spans a **pass band**.  The frequency ranges that no branch reaches
are the **band gaps**: no wave of that frequency can travel along the rotor.

Every branch is classified as forward or backward whirl from its own
eigenvector (the whirl index), not from the order of the frequencies — so the
labels stay correct at any speed, including where branches cross.

In [ ]:
from rotorwave import OmegaKSolver
from rotorwave.plotting import plot_dispersion, set_style

set_style()
dispersion = OmegaKSolver(cell).solve(spin_rpm=6000.0, n_points=61, n_branches=8)

print(dispersion.summary('forward'))
print(dispersion.summary('backward'))

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
plot_dispersion(dispersion, ax=ax, f_max=5000.0);

Comparing with the rotor at rest shows the gyroscopic effect: the forward gap
moves up, the backward gap moves down, and the width stays practically the
same.  This is the central quantitative claim of the paper — the stop band can
be *tuned* with the rotating speed without being destroyed.

In [ ]:
at_rest = OmegaKSolver(cell).solve(spin_rpm=0.0, n_points=61, n_branches=8)

for label, d, kind in ((' 0 rpm', at_rest, 'both'),
                       ('6000 rpm fw', dispersion, 'forward'),
                       ('6000 rpm bw', dispersion, 'backward')):
    gap = d.band_gaps(kind)[0]
    print(f'{label:>12s}: {gap.start_hz:7.1f} - {gap.stop_hz:7.1f} Hz'
          f'  (width {gap.width_hz:.1f} Hz)')

## 3. The $k(\omega)$ problem and the frequency response

Now the other way round: prescribe a real frequency and compute the wavenumbers.
They come out **complex** — the real part describes the propagation, the
imaginary part the attenuation per cell.  This is what tells you *how strongly*
a rotor rejects a frequency, which the $\omega(k)$ problem cannot say.

Below the dispersion, the receptance of the finite 11-disk rotor.  The quiet
regions must line up with the stop bands: that agreement between an infinite
periodic medium and a finite rotor is the physical validation of the whole
analysis.

In [ ]:
from rotorwave import KOmegaSolver, ReceptanceSolver
from rotorwave.plotting import plot_komega_dispersion

frequencies = np.arange(10.0, 5001.0, 10.0)
complex_k = KOmegaSolver(cell).solve(frequencies, spin_rpm=6000.0)
receptance = ReceptanceSolver(rotor.full_rotor(elements_per_cell=10)).compute(
    frequencies, spin_rpm=6000.0)

fig, axes = plot_komega_dispersion(complex_k, receptance, f_max=5000.0)

Directional FRFs (Lee, 1991) split the response into its forward and backward
whirl contributions, so the resonances of each precession direction can be read
separately — the measurable counterpart of the forward/backward branches
above.

In [ ]:
forward = receptance.resonances('forward')
backward = receptance.resonances('backward')
print('last forward  resonances:', np.round(forward[forward < 2500][-3:], 0))
print('last backward resonances:', np.round(backward[backward < 2500][-3:], 0))

## 4. Analyses beyond the paper

### 4.1 How the band gaps move with the rotating speed

The paper states that the gap shifts with speed but keeps its width.  Sweeping
the speed and tracking the gap edges turns that qualitative statement into
numbers.

In [ ]:
from rotorwave import band_gap_map
from rotorwave.plotting import plot_band_gap_map

speeds = np.arange(0.0, 12001.0, 2000.0)
gap_map = band_gap_map(rotor.unit_cell(), speeds, n_points=31,
                       n_branches=8, f_max=5000.0)
print(gap_map.summary())

ax = plot_band_gap_map(gap_map, f_max=5000.0);

### 4.2 Attenuation per cell

Inside a stop band no wave propagates, but the response does not vanish
abruptly: it decays cell by cell.  The attenuation says how many dB a wave
loses every time it crosses one period of the rotor; multiplied by the number
of cells, it is the isolation the whole rotor provides.

In [ ]:
from rotorwave import attenuation_report
from rotorwave.plotting import plot_attenuation

attenuation = attenuation_report(cell, frequencies, spin_rpm=6000.0,
                                 n_cells=rotor.n_cells)
print(f"peak attenuation across the whole rotor: "
      f"{attenuation['forward_db_total'].max():.0f} dB")

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
plot_attenuation(attenuation, ax=ax, f_max=5000.0);

## 5. Your own rotor

Any geometry: change the parameters of `PeriodicRotor`.  Shorter cells push the
Bragg limit — and with it the first band gap — to higher frequencies, which is
the design rule the paper derives.

In [ ]:
from rotorwave import STEEL, PeriodicRotor

my_rotor = PeriodicRotor(
    shaft_length=0.96,        # total shaft length [m]
    shaft_diameter=20e-3,     # shaft outer diameter [m]
    n_cells=3,                # number of disks
    disk_diameter=200e-3,     # disk outer diameter [m]
    disk_thickness=12e-3,     # disk thickness [m]
    material=STEEL,
    elements_per_cell=40,
)
print(my_rotor.summary())

d = OmegaKSolver(my_rotor.unit_cell()).solve(spin_rpm=1500.0, n_points=61,
                                             n_branches=6)
print(d.summary('forward'))